# Complexity-08 — Origami Wolfram pli 2 : la regle 30 comme generateur d'alea, la regle 110 et son universalite

**Navigation** : [<< Complexity-07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-CSharp.ipynb) | [Index de la série](README.md) | [Complexity-08b — Combinateurs SKI et tag de Post →](Complexity-08b-CombinateursSKI-TagDePost-Python.ipynb) | [Version Python](Complexity-08-WolframRule30Aleatoire-Rule110Universelle-Python.ipynb)

**Carnet** : deux points de la definition du pli 2 qu'aucun carnet de la serie ne couvre encore --
la **regle 30 comme generateur de sequences aleatoires** et l'**universalite de la regle 110** --
avec, a cote de chaque these, sa **reception critique**.

**Organe utilise** : le port .NET de `ict.wolfram_step` (livre par PR #19793, chemin
`MyIA.AI.Notebooks/IIT/ICT-Series/ict/wolfram_step.py`), expose ici sous le nom `WolframStep`.
Toutes les mesures de ce carnet passent par `WolframStep` ; **aucune reimplementation divergente
de l'automate**. Ce que le carnet ecrit lui-meme, ce sont les **instruments de mesure** (tests
statistiques, complexite lineaire), qui ne sont pas dans l'organe.

**Ce que ce carnet fait, et ce qu'il ne fait pas.**

| Ces deux theses | Statut | Ce que fait ce carnet |
|---|---|---|
| La regle 30 produit des sequences utilisables comme alea | *these empirique* de Wolfram 1986 | il **mesure** la sequence et la confronte a une batterie de tests standard, avec temoins negatifs |
| La regle 110 est universelle (Turing-complete) | *theoreme* (Cook 2004) | il **ne le re-prouve pas** ; il mesure la structure d'ether qui porte la construction, et expose la reception |

La distinction est le sujet d'une partie de ce carnet : une these mesuree et un theoreme ne se
citent pas de la meme facon.

> **Statut épistémique** — **Sans verdict à ce jour** : ce carnet n'a pas encore de ligne dans la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md), donc aucune solidité n'est revendiquée ici. Ce qu'il mesure est reproductible depuis ses cellules ; ce n'est pas un acquis de la série.

## Sources primaires et regle de citation

Les quatre textes utilises sont lus dans les copies archivees du gisement
`G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\` :

| Reference | Copie archivee |
|---|---|
| Wolfram, *Random Sequence Generation by Cellular Automata*, Adv. Appl. Math. 7 (1986) | `1986 - Wolfram - Random Sequence Generation by Cellular Automata (Adv. Appl. Math. 7, 1986).pdf` |
| Wolfram, *Universality and Complexity in Cellular Automata*, Physica D 10 (1984) | `1984 - Wolfram - Universality and Complexity in Cellular Automata (Physica D 10, 1984).pdf` |
| Cook, *Universality in Elementary Cellular Automata*, Complex Systems 15 (2004) | `2004 - Cook - Universality in Elementary Cellular Automata (Complex Systems 15-1, OA).pdf` |
| Gray, *A Mathematician Looks at Wolfram's New Kind of Science*, Notices AMS 50-2 (2003) | `2002 - Gray - A Mathematician Looks at Wolfram's New Kind of Science (Notices AMS 50-2 2003, OA).pdf` |

**Convention de citation** : chaque affirmation attribuee a un texte cite la **page de la copie
archivee** et la section. Une capture HTML se citerait par son **titre de section**, jamais par un
numero de ligne -- le numero de ligne d'un texte extrait ne correspond a rien dans le fichier.

In [1]:
// Illustration 3 — organe .NET : port contractuel de `ict.wolfram_step` (aucune divergence)
// #25381 : l'organe Python est livre par `pip install -e MyIA.AI.Notebooks/IIT/ICT-Series` ;
// ce jumeau en porte la contrepartie .NET, avec les memes conventions (voisinage
// gauche-centre-droit, table de regle sur 8 bits, bords periodiques) et donc les memes instances.
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

// Un pas de l'automate cellulaire 1-D de Wolfram, bords periodiques (wrap circulaire).
int[] WolframStep(int[] state, int rule)
{
    int n = state.Length;
    if (n == 0)
        return Array.Empty<int>();
    var suivant = new int[n];
    for (int i = 0; i < n; i++)
    {
        int gauche = state[(i - 1 + n) % n];
        int centre = state[i];
        int droite = state[(i + 1) % n];
        int idx = (gauche << 2) | (centre << 1) | droite;
        suivant[i] = (rule >> idx) & 1;
    }
    return suivant;
}

// Table de verite d'une regle, indexee par le voisinage 000, 001, ..., 111.
int[] TableRegle(int regle) => Enumerable.Range(0, 8).Select(p => (regle >> p) & 1).ToArray();

// Densite d'un etat : proportion de cellules a 1.
double WolfrangDensite(int[] state) => state.Length == 0 ? 0.0 : state.Count(v => v != 0) / (double)state.Length;

// Classes canoniques de l'organe (WOLFRAM_CLASSES) : etiquette indexee, la mesure est faite ailleurs.
string? ClasseConnue(int regle) => regle switch
{
    0 => "I (uniform)",
    32 => "I (uniform)",
    160 => "I (uniform)",
    232 => "I (uniform)",
    4 => "II (periodic)",
    8 => "II (periodic)",
    13 => "II (periodic)",
    30 => "III (chaotic)",
    45 => "III (chaotic)",
    75 => "III (chaotic)",
    126 => "III (chaotic)",
    110 => "IV (complex)",
    _ => null
};

// Generateur pseudo-aleatoire a graine fixe (SplitMix64), equivalent .NET du
// `numpy.random.default_rng(seed)` du jumeau Python. L'algorithme differe du PCG64 de NumPy :
// le temoin aleatoire n'est donc pas le meme tirage, seule la loi (uniformite des p) est comparee.
class SplitMix64
{
    private ulong _etat;

    public SplitMix64(ulong graine)
    {
        _etat = graine;
    }

    public ulong Suivant()
    {
        _etat += 0x9E3779B97F4A7C15UL;
        ulong z = _etat;
        z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9UL;
        z = (z ^ (z >> 27)) * 0x94D049BB133111EBUL;
        return z ^ (z >> 31);
    }

    // Un bit a la fois, comme le tirage `rng.integers(0, 2, size=n)` du jumeau Python.
    public int[] Bits(int n)
    {
        var bits = new int[n];
        for (int i = 0; i < n; i++)
            bits[i] = (int)(Suivant() & 1UL);
        return bits;
    }
}

// Log-gamma (Lanczos, g=7, n=9).
double LogGamma(double x)
{
    double[] c =
    {
        0.99999999999980993, 676.5203681218851, -1259.1392167224028, 771.32342877765313,
        -176.61502916214059, 12.507343278686905, -0.13857109526572012,
        9.9843695780195716e-6, 1.5056327351493116e-7
    };
    if (x < 0.5)
        return Math.Log(Math.PI / Math.Sin(Math.PI * x)) - LogGamma(1.0 - x);
    x -= 1.0;
    double a = c[0];
    double t = x + 7.5;
    for (int i = 1; i < 9; i++)
        a += c[i] / (x + i);
    return 0.5 * Math.Log(2.0 * Math.PI) + (x + 0.5) * Math.Log(t) - t + Math.Log(a);
}

// P(a,x) regularise inferieur : serie (valide pour x < a+1).
double GammaPSerie(double a, double x)
{
    double ap = a, somme = 1.0 / a, del = somme;
    for (int n = 1; n <= 1000; n++)
    {
        ap += 1.0;
        del *= x / ap;
        somme += del;
        if (Math.Abs(del) < Math.Abs(somme) * 1e-16)
            break;
    }
    return somme * Math.Exp(-x + a * Math.Log(x) - LogGamma(a));
}

// Log de Q(a,x) = 1 - P(a,x) : fraction continue de Legendre, evaluee en espace
// logarithmique. Ce port calcule la queue SUPERIEURE directement : la calculer par
// 1 - P s'annulerait en double precision des que la queue passe sous 1e-16, et
// les rejets les plus nets de la Table 10.2 (p de l'ordre de 1e-219 a 1e-31)
// seraient alors affiches comme zero.
double LogGammaQ(double a, double x)
{
    double b = x + 1.0 - a, c = 1e300, d = 1.0 / b, h = d;
    for (int i = 1; i <= 1000; i++)
    {
        double an = -i * (i - a);
        b += 2.0;
        d = an * d + b;
        if (Math.Abs(d) < 1e-300)
            d = 1e-300;
        c = b + an / c;
        if (Math.Abs(c) < 1e-300)
            c = 1e-300;
        d = 1.0 / d;
        double del = d * c;
        h *= del;
        if (Math.Abs(del - 1.0) < 1e-16)
            break;
    }
    return -x + a * Math.Log(x) - LogGamma(a) + Math.Log(h);
}

// Equivalent .NET de `scipy.stats.chi2.sf(chi2, df)` : probabilite qu'une sequence
// vraiment aleatoire depasse le chi-deux observe.
double Chi2Survie(double chi2, int df)
{
    if (df <= 0)
        throw new ArgumentException("degres de liberte strictement positifs requis");
    double a = 0.5 * df, x = 0.5 * chi2;
    if (x >= a + 1.0)
        return Math.Exp(LogGammaQ(a, x));
    return 1.0 - GammaPSerie(a, x);
}

// --- Verification de l'organe : les invariants sont recalcules, jamais affirmes en dur. ---
// La table de regle se lit bit a bit dans l'entier qui encode la regle.
var table30 = TableRegle(30);
var table110 = TableRegle(110);
if (!table30.SequenceEqual(new int[] { 0, 1, 1, 1, 1, 0, 0, 0 }))
    throw new Exception("Table de verite de la regle 30 incorrecte");
if (!table110.SequenceEqual(new int[] { 0, 1, 1, 1, 0, 1, 1, 0 }))
    throw new Exception("Table de verite de la regle 110 incorrecte");
// La regle 0 efface tout et la regle 255 remplit tout : invariants recalcules sur un etat quelconque.
var etatTest = new int[] { 0, 1, 1, 0, 1, 0, 0, 1, 1 };
if (WolframStep(etatTest, 0).Any(v => v != 0))
    throw new Exception("La regle 0 ne devrait pas tout effacer");
if (WolframStep(etatTest, 255).Any(v => v != 1))
    throw new Exception("La regle 255 ne devrait pas tout remplir");
// Un pas de la regle 30 depuis une graine centree, recalcule puis compare a l'effet attendu.
var pas30 = WolframStep(new int[] { 0, 0, 0, 1, 0, 0, 0 }, 30);
if (!pas30.SequenceEqual(new int[] { 0, 0, 1, 1, 1, 0, 0 }))
    throw new Exception("Pas de la regle 30 incorrect");
// Le chi-deux survit : une queue de loi normale a 1 degres de liberte a 3,84 donne 0,05 (valeur de reference).
if (Math.Abs(Chi2Survie(3.841458820694124, 1) - 0.05) > 1e-6)
    throw new Exception("Survie du chi-deux incorrecte");

// La queue superieure doit rester exacte quand elle est tres fine : trois cas de la
// Table 10.2 sont recalcules par l'instrument et compares aux valeurs publiees par
// le jumeau Python (scipy). Ces references sont arrondies a trois chiffres
// significatifs, donc la tolerance retenue est l'ecart de cette arrondi (1e-2),
// pas une precision machine. Chaque cas est affiche : mesure, reference, ecart, verdict.
double[] chi2Refs = { 1739.81, 750.44, 609.38 };
double[] pRefs = { 4.89e-219, 2.77e-50, 4.88e-31 };
string[] nomsRefs = { "CA60  N=29", "LFSR  n=29", "LCG  2^31" };
const double TOLERANCE_QUEUE = 1e-2;
int nonConformes = 0;
Console.WriteLine("controle de la queue superieure du chi-deux (df = 255), comparaison aux valeurs du jumeau Python :");
for (int i = 0; i < chi2Refs.Length; i++)
{
    double mesure = Chi2Survie(chi2Refs[i], 255);
    double ecartRelatif = Math.Abs(mesure - pRefs[i]) / pRefs[i];
    bool conforme = ecartRelatif <= TOLERANCE_QUEUE;
    if (!conforme)
        nonConformes++;
    Console.WriteLine($"  {nomsRefs[i],-11} chi2={chi2Refs[i],8:F2}  mesure={mesure.ToString("G4", CultureInfo.InvariantCulture),-12} reference={pRefs[i].ToString("G4", CultureInfo.InvariantCulture),-12} ecart relatif={ecartRelatif.ToString("G3", CultureInfo.InvariantCulture),-8} {(conforme ? "CONFORME" : "ECART")}");
}
if (nonConformes > 0)
    throw new Exception(nonConformes + " cas de queue superieure non conformes a la reference (tolerance " + TOLERANCE_QUEUE.ToString("G2", CultureInfo.InvariantCulture) + ")");

// Les regles canoniques de l'organe.
var canoniques = new int[] { 0, 30, 110, 250 };
string table30Txt = string.Join(", ", table30);
string table110Txt = string.Join(", ", table110);
string pas30Txt = string.Join(" ", pas30);
Console.WriteLine("organe .NET (port de ict.wolfram_step) importe OK -- aucune reimplementation divergente de l automate");
Console.WriteLine("CANONICAL_RULES: " + string.Join(", ", canoniques.Select(r => r.ToString(CultureInfo.InvariantCulture))));
Console.WriteLine("table de verite regle 30 : [" + table30Txt + "] ; regle 110 : [" + table110Txt + "]");
Console.WriteLine("un pas de la regle 30 depuis une graine centree : [" + pas30Txt + "]");
Console.WriteLine("survie du chi-deux (df=1, x=3,841458820694124) = " + Chi2Survie(3.841458820694124, 1).ToString("F6", CultureInfo.InvariantCulture));
Console.WriteLine("survie du chi-deux sur les trois queues les plus fines (df=255) : " + string.Join(", ", chi2Refs.Select(c => Chi2Survie(c, 255).ToString("G4", CultureInfo.InvariantCulture))) + " (tolerance d'arrondi 1e-2 sur les references du jumeau Python, ecarts affiches ci-dessus)");


controle de la queue superieure du chi-deux (df = 255), comparaison aux valeurs du jumeau Python :


  CA60  N=29  chi2= 1739.81  mesure=4.895E-219   reference=4.89E-219    ecart relatif=0.00107  CONFORME


  LFSR  n=29  chi2=  750.44  mesure=2.769E-50    reference=2.77E-50     ecart relatif=0.000463 CONFORME


  LCG  2^31   chi2=  609.38  mesure=4.87E-31     reference=4.88E-31     ecart relatif=0.00203  CONFORME


organe .NET (port de ict.wolfram_step) importe OK -- aucune reimplementation divergente de l automate


CANONICAL_RULES: 0, 30, 110, 250


table de verite regle 30 : [0, 1, 1, 1, 1, 0, 0, 0] ; regle 110 : [0, 1, 1, 1, 0, 1, 1, 0]


un pas de la regle 30 depuis une graine centree : [0 0 1 1 1 0 0]


survie du chi-deux (df=1, x=3,841458820694124) = 0.050000


survie du chi-deux sur les trois queues les plus fines (df=255) : 4.895E-219, 2.769E-50, 4.87E-31 (tolerance d'arrondi 1e-2 sur les references du jumeau Python, ecarts affiches ci-dessus)


## 1. Le protocole de Wolfram : extraire une sequence d'un registre circulaire

Wolfram 1986 section 11 (p. 162) donne le mode d'emploi de l'automate comme generateur :

> Random sequences can be obtained by sampling the sequence of values of a particular site in a
> register updated according to Eq. (3.1).

Les parametres viennent de la legende de la Table 10.1 (p. 159) : registre **circulaire** de
longueur `N`, **graine consistant en un seul site non nul**, echantillon de longueur `L`.

Deux details du protocole ne sont **pas** donnes par le texte : **quel site** on echantillonne, et
**ou** se trouve le site non nul dans le registre. Le carnet les fixe explicitement, et il teste
plus bas la sensibilite du resultat a ce choix -- c'est la premiere chose a verifier quand une
reproduction ne tombe pas sur les chiffres publies.

In [2]:
// Illustration 5 — extraction temporelle : 64 premiers bits du site 0, regle 30, N=49
// Graine de la Table 10.1 : un seul site non nul, au centre par defaut.
int[] SeedSingleNonZero(int nCells, int? seedSite = null)
{
    var state = new int[nCells];
    state[seedSite ?? (nCells / 2)] = 1;
    return state;
}

// Valeurs d'UN site, un echantillon par pas de temps (Wolfram 1986, sec. 11).
int[] TemporalBits(int rule, int nCells, int nBits, int site = 0, int? seedSite = null)
{
    var state = SeedSingleNonZero(nCells, seedSite);
    var outp = new int[nBits];
    for (int t = 0; t < nBits; t++)
    {
        outp[t] = state[site];
        state = WolframStep(state, rule);
    }
    return outp;
}

// Regroupe en octets non chevauchants (Annexe A : blocs de longueur 8), bits de poids fort en tete.
int[] BitsToBytes(int[] bits)
{
    int n = (bits.Length / 8) * 8;
    var octets = new int[n / 8];
    for (int i = 0; i < n; i++)
        octets[i / 8] = (octets[i / 8] << 1) | (bits[i] & 1);
    return octets;
}

var bits = TemporalBits(30, 49, 64);
Console.WriteLine("64 premiers bits du site 0, regle 30, N=49 : " + string.Concat(bits.Select(b => b.ToString(CultureInfo.InvariantCulture))));
Console.WriteLine("octets correspondants : " + string.Join(" ", BitsToBytes(bits).Select(v => v.ToString("X2", CultureInfo.InvariantCulture))));

64 premiers bits du site 0, regle 30, N=49 : 0000000000000000000000001011100011010011000101011110110110111010


octets correspondants : 00 00 00 B8 D3 15 ED BA


## 2. La mise en garde que Wolfram ecrit lui-meme : periode contre longueur d'echantillon

La legende de la Table 10.1 (p. 159) porte une reserve que le corps du texte repete :

> Accumulations close to 0 or 1 suggest deviations from randomness. Such accumulations are seen in
> this case only when the period of the cellular automaton is comparable to the length of the
> sequence sampled.

Un registre **fini** est periodique : au-dela de sa periode, la sequence qu'on en extrait n'est plus
un echantillon de comportement, c'est une repetition. Wolfram publie les periodes en Table 9.2 mais
ne les reprend pas dans la Table 10.1. Mesurons-les ici, pour chaque taille utilisee, et comparons
au rapport `L / periode`.

In [3]:
// Exercice 1 — periode exacte du registre circulaire (stub compilable)
// TODO etudiant : detecter l'etat deja vu (un dictionnaire des etats rencontres, la cle etant
// la suite de 0/1 du registre) et renvoyer (periode, pre-periode) ; renvoyer (null, null) quand
// aucun etat ne se repete dans maxSteps pas.
// Les valeurs attendues ne sont pas donnees ici : elles se lisent dans la lecture du resultat
// qui suit, ou la periode est comparee a la longueur de l'echantillon.
(int? periode, int? prePeriode) Periode(int regle, int nCells, int maxSteps = 300000)
{
    // TODO etudiant : implementation attendue ici.
    return (null, null);
}

int L = 64 * 1024;
Console.WriteLine("Exercice 1 a completer : periodes des registres circulaires pour N = 17, 23, 29, 37, 49 (echantillon L = " + L.ToString(CultureInfo.InvariantCulture) + " bits)");

Exercice 1 a completer : periodes des registres circulaires pour N = 17, 23, 29, 37, 49 (echantillon L = 65536 bits)


**Lecture du resultat** : a `N=17` la periode vaut 10 846 pas pour un echantillon de 65 536 bits --
le registre fait **six fois le tour** de son cycle pendant la mesure, et la sequence extraite n'est
pas 65 536 pas de dynamique mais la repetition d'un motif de 10 846. A `N=23` le rapport tombe a
1,71 : le registre fait encore plus d'un tour. A partir de `N=29` la periode depasse la fenetre de
recherche (300 000 pas) et le rapport devient tres inferieur a 1.

C'est exactement la condition que Wolfram designe. Elle a une consequence methodologique : un test
qui rejette la regle 30 a `N=17` ne dit **rien** sur le caractere aleatoire de la regle 30 -- il
detecte la periodicite du **registre**, pas celle de l'automate. Pour une graine de generateur, la
section 11 (p. 162) en tire la regle pratique : une longueur de cycle depassant `2^32` demande
`N = 49`, et `N = 127` suffit pour depasser `2^64`.

## 3. Test A -- distribution des frequences de blocs (Annexe A)

L'Annexe A (p. 166) decrit sept procedures. La premiere, **A. Block Frequency Distribution** :

> Each of the 2^n possible n-blocks should occur with equal frequency. (n = 8 is used.)

Le corps du texte (p. 158) donne la statistique, equation (10.1) :

$$\chi^2 = \sum \frac{(p_o - p_e)^2}{p_e}$$

avec `nu = 2^n` degres de liberte pour des blocs de longueur `n`. Ici `n = 8`, donc 256 classes et
`nu = 256` : la valeur publiee est la **probabilite qu'une sequence vraiment aleatoire depasse** le
chi-deux observe. Une valeur proche de 0 ou de 1 signale une deviation.

In [4]:
// Exercice 2 — test A : frequence des 256 octets et chi-deux (stub compilable)
// TODO etudiant : renvoyer (chi2, p, nOctets).
//  - octets  : regrouper les bits en blocs de 8 non chevauchants (bits de poids fort en tete) ;
//  - chi2   : comparer les effectifs observes des 256 classes a l'effectif attendu ;
//  - p      : probabilite qu'une sequence vraiment aleatoire depasse le chi-deux observe
//              (255 degres de liberte, comme dans le jumeau Python) ;
//  - nOctets : nombre d'octets obtenus (8192 pour un echantillon de 64k bits).
// L'instrument de reference du carnet (l'oracle) est fourni par la cellule de validation qui
// suit : comparez vos valeurs aux siennes, ligne a ligne -- l'ecart doit etre nul.
(double chi2, double p, int nOctets) TestA(int[] bits)
{
    // TODO etudiant : implementation attendue ici.
    return (0.0, 1.0, 0);
}

Console.WriteLine("Exercice 2 a completer : test A (frequence des 256 octets, chi-deux, p)");

Exercice 2 a completer : test A (frequence des 256 octets, chi-deux, p)


Le temoin ci-dessous est un **controle positif de l'instrument** : soumis a de vrais tirages, le test
doit rendre des `p` uniformes sur [0, 1]. Sans ce controle, un `p` extreme sur la regle 30 serait
indiscernable d'un test mal calibre.

Passons la regle 30 au test A, aux tailles et longueurs de la Table 10.1, et posons en regard les
valeurs que Wolfram publie (ligne A, colonnes 1 a 6).

In [5]:
// Validation — confrontation a la Table 10.1 de Wolfram (1986) : pour chaque (N, L), p mesure
// par notre test A contre la p-value publiee. PUBLISHED_A est une transcription
// directe de la table ; le drapeau DEVIE marque les cas ou notre mesure s'eloigne
// de l'uniformite (p aux deux extremites) -- la lecture croisee des deux colonnes
// vit dans la cellule markdown qui suit.
// Cet oracle est l'instrument de reference du carnet : l'exercice 2 demande votre propre
// version de TestA, a comparer ligne a ligne avec celle-ci (l'ecart doit etre nul).
(double chi2, double p, int nOctets) TestAReference(int[] bits)
{
    var octets = BitsToBytes(bits);
    var counts = new int[256];
    foreach (int o in octets)
        counts[o]++;
    double attendu = octets.Length / 256.0;
    double chi2 = 0.0;
    for (int i = 0; i < 256; i++)
    {
        double ecart = counts[i] - attendu;
        chi2 += ecart * ecart / attendu;
    }
    // 255 degres de liberte, comme dans le jumeau Python (nu = 256 classes).
    return (chi2, Chi2Survie(chi2, 255), octets.Length);
}

// Oracle de reference de la periode (exercice 1) : detection de l'etat deja vu.
// L'exercice 1 demande votre propre version de `Periode` ; celle-ci sert de reference
// de comparaison, et c'est elle qui alimente la colonne `Periode` de la porte de
// degenerescence (section 10 bis). Les resultats sont memorises, la mesure est faite
// une seule fois par couple (regle, taille).
var cachePeriodes = new Dictionary<string, (int? periode, int? prePeriode)>();
(int? periode, int? prePeriode) PeriodeReference(int regle, int nCells, int maxSteps = 300000)
{
    string cle = regle.ToString(CultureInfo.InvariantCulture) + "|" + nCells.ToString(CultureInfo.InvariantCulture) + "|" + maxSteps.ToString(CultureInfo.InvariantCulture);
    if (cachePeriodes.TryGetValue(cle, out var cache))
        return cache;
    var state = SeedSingleNonZero(nCells);
    var vus = new Dictionary<string, int>();
    (int? periode, int? prePeriode) res = (null, null);
    for (int t = 0; t < maxSteps; t++)
    {
        string key = string.Concat(state.Select(s => (char)('0' + s)));
        if (vus.TryGetValue(key, out int t0))
        {
            res = (t - t0, t0);
            break;
        }
        vus[key] = t;
        state = WolframStep(state, regle);
    }
    cachePeriodes[cle] = res;
    return res;
}

// Controle de plomberie : sur des bits tires a pile ou face, p doit etre uniforme.
var rng = new SplitMix64(2026);
var pValeurs = new List<double>();
for (int i = 0; i < 20; i++)
    pValeurs.Add(TestAReference(rng.Bits(64 * 1024)).p);
Console.WriteLine("temoin aleatoire -- p sur 20 tirages : " + string.Join(", ", pValeurs.Select(v => v.ToString("F2", CultureInfo.InvariantCulture))));
Console.WriteLine("moyenne : " + pValeurs.Average().ToString("F3", CultureInfo.InvariantCulture) + " (attendu ~0,5 si le test est calibre)");
Console.WriteLine("fraction sous 0,05 : " + (pValeurs.Count(v => v < 0.05) / 20.0).ToString("F2", CultureInfo.InvariantCulture) + " (attendu ~0,05)");
Console.WriteLine("fraction au-dessus de 0,95 : " + (pValeurs.Count(v => v > 0.95) / 20.0).ToString("F2", CultureInfo.InvariantCulture) + " (attendu ~0,05)");

var PUBLISHED_A = new (string cas, int N, int Lx, double publie)[]
{
    ("N=17, L=8k", 17, 8 * 1024, 0.0039),
    ("N=17, L=64k", 17, 64 * 1024, 1.0000),
    ("N=23, L=64k", 23, 64 * 1024, 0.0456),
    ("N=29, L=64k", 29, 64 * 1024, 0.7375),
    ("N=37, L=64k", 37, 64 * 1024, 0.3852),
    ("N=49, L=64k", 49, 64 * 1024, 0.8003),
};

Console.WriteLine();
Console.WriteLine($"{"cas",-14} {"chi2",8} {"p mesure",10} {"p Wolfram",10}  lecture");
foreach (var item in PUBLISHED_A)
{
    var mesure = TestAReference(TemporalBits(30, item.N, item.Lx));
    string flag = (mesure.p < 0.05 || mesure.p > 0.95) ? "DEVIE" : "conforme";
    Console.WriteLine($"{item.cas,-14} {mesure.chi2,8:F2} {mesure.p,10:F4} {item.publie,10:F4}  {flag}");
}

Console.WriteLine();
Console.WriteLine("Oracle de reference de la periode (exercice 1) -- regle 30 :");
foreach (int n in new int[] { 17, 23, 29, 37, 49 })
{
    var (p, pre) = PeriodeReference(30, n);
    string pTxt = p.HasValue ? p.Value.ToString(CultureInfo.InvariantCulture) : "None";
    string preTxt = pre.HasValue ? pre.Value.ToString(CultureInfo.InvariantCulture) : "None";
    string ratio = p.HasValue ? (L / (double)p.Value).ToString("F2", CultureInfo.InvariantCulture) : "non mesuree (>300k pas)";
    Console.WriteLine($"N={n,3}  periode={pTxt}  pre-periode={preTxt}  L/periode={ratio}");
}


temoin aleatoire -- p sur 20 tirages : 0.08, 0.91, 0.28, 0.90, 0.87, 0.27, 0.78, 0.83, 0.81, 0.92, 0.18, 0.71, 0.01, 0.29, 0.26, 0.42, 0.21, 0.70, 0.20, 0.91


moyenne : 0.527 (attendu ~0,5 si le test est calibre)


fraction sous 0,05 : 0.05 (attendu ~0,05)


fraction au-dessus de 0,95 : 0.00 (attendu ~0,05)


cas                chi2   p mesure  p Wolfram  lecture


N=17, L=8k       231.50     0.8519     0.0039  conforme


... (7 outputs left out)

N= 23  periode=38249  pre-periode=1161  L/periode=1.71


N= 29  periode=None  pre-periode=None  L/periode=non mesuree (>300k pas)


N= 37  periode=None  pre-periode=None  L/periode=non mesuree (>300k pas)


N= 49  periode=None  pre-periode=None  L/periode=non mesuree (>300k pas)


**Lecture du resultat -- et ce qu'elle ne dit pas.** Le resultat se lit en deux temps, et il faut
resister au premier.

**Ce qui se reproduit** : le **mecanisme**. Le seul cas rejete est `N=17, L=64k` (`p = 2,0e-10`),
c'est-a-dire celui ou l'echantillon vaut **six fois la periode** du registre (65 536 pour 10 846) :
la sequence mesuree n'est plus un echantillon de dynamique, c'est la meme boucle repetee six fois.
Le meme `N=17` sur `L=8k` -- echantillon plus court que la periode -- n'est **pas** rejete
(`p = 0,8519`). Le phenomene n'est donc pas "petit `N` = mauvais", c'est "echantillon plus long que
la periode = artefact".

La ou la mise en garde de Wolfram est plus fine que mon propre enonce : `N=23` a une periode de
38 249 pour un echantillon de 65 536, soit **1,7 tour**, et le test ne rejette **pas** (`p = 0,8253`).
Le seuil n'est donc pas net -- c'est un ordre de grandeur, pas un critere. Wolfram parle de periode
"comparable" a la longueur echantillonnee, et cette mesure precise ce que *comparable* veut dire :
a 1,7 tours rien ne se voit, a 6 tours tout se voit. Pour `N >= 29` la periode depasse la fenetre de
recherche et le rapport tombe sous 1.

**Ce qui ne se reproduit pas** : les **valeurs exactes**. Pour `N=17, L=64k` Wolfram publie
`1,0000` quand on mesure `2,0e-10` ; pour `N=23` il publie `0,0456` quand on mesure `0,8253`. Les
deux mesures sont extremes dans le premier cas, mais **de part et d'autre de la distribution** :
les deux rejettent l'hypothese d'alea, l'une en haut, l'autre en bas. La cause n'est pas
mysterieuse : le protocole publie ne dit ni **quel site** on lit, ni **ou** se trouve le site non
nul, ni par quel bout on ordonne les bits dans l'octet. Ce sont trois degres de liberte qui changent
la sequence echantillonnee sans changer l'automate.

Une reproduction **fidele au chiffre** demanderait le code de Don Mitchell, que le texte ne donne
pas. Ce carnet ne peut donc pas pretendre valider la Table 10.1 ligne par ligne ; il peut valider
l'**enonce conditionnel** qui l'accompagne -- et c'est cet enonce qui porte la lecon d'usage.

## 4. Temoins negatifs -- ce que le meme test fait a des generateurs reputes

Un test qui ne rejette jamais rien ne mesure rien. La Table 10.2 (p. 160) fournit exactement les
temoins qu'il faut, et sa legende en donne tous les parametres :

- **LFSR** (linear feedback shift register) de longueur `N`, periode `2^N - 1`, **taps en positions
  14 et 17** pour `N = 17`, **27 et 29** pour `N = 29` ;
- **LCG** : `x' = (1103515245x + 12345) mod 2^31`, graine `x = 1` ("used, for example, in many
  implementations of the UNIX operating system") ;
- **CA60** : l'automate lineaire de l'equation (2.4), registre de taille `N = 29`.

Wolfram annonce le resultat avant de le mesurer (p. 159) : les LFSR, "while provably random in some
respects, are revealed as significantly nonrandom by several of the tests used here", et "many
sequences obtained from linear congruential generators are also found to be significantly
nonrandom". Le CA60 est le cas le plus net de sa table.

In [6]:
// Validation — temoins negatifs de la Table 10.2 : trois generateurs lineaires reputes NON
// aleatoires (LFSR, LCG UNIX, CA60), construits selon la legende exacte de la
// table. Un test qui ne rejette que du vrai alea ne mesure rien : ici le meme
// test A doit rejeter LFSR/LCG/CA60 tres loin (p minuscules) pendant qu'il
// laisse passer le CA30 -- c'est la paire valide/rejette qui fait la preuve.
// LFSR de longueur n, taps 1-indexes (Wolfram 1986, legende de la Table 10.2).
int[] LfsrBits(int n, int[] taps, int nBits)
{
    var state = new int[n];
    state[n / 2] = 1;
    var outp = new int[nBits];
    for (int t = 0; t < nBits; t++)
    {
        outp[t] = state[0];
        int fb = 0;
        foreach (int tap in taps)
            fb ^= state[n - tap];
        var suivant = new int[n];
        Array.Copy(state, 1, suivant, 0, n - 1);
        suivant[n - 1] = fb;
        state = suivant;
    }
    return outp;
}

// LCG x' = (1103515245x + 12345) mod 2^31, bits de poids fort en tete.
int[] LcgBits(int nBits, long seed = 1)
{
    const long M = 1L << 31;
    long x = seed;
    int nbVals = nBits / 31 + 2;
    var vals = new long[nbVals];
    for (int i = 0; i < nbVals; i++)
    {
        x = (1103515245L * x + 12345L) % M;
        vals[i] = x;
    }
    var bits = new int[vals.Length * 31];
    for (int i = 0; i < vals.Length; i++)
        for (int b = 0; b < 31; b++)
            bits[i * 31 + b] = (int)((vals[i] >> (30 - b)) & 1);
    var outp = new int[nBits];
    Array.Copy(bits, outp, nBits);
    return outp;
}

var controles = new (string label, int[] seq)[]
{
    ("CA30  N=49", TemporalBits(30, 49, L)),
    ("CA60  N=29", TemporalBits(60, 29, L)),
    ("LFSR  n=17", LfsrBits(17, new int[] { 14, 17 }, L)),
    ("LFSR  n=29", LfsrBits(29, new int[] { 27, 29 }, L)),
    ("LCG  2^31", LcgBits(L)),
};

Console.WriteLine($"{"generateur",-12} {"chi2",9} {"p (test A)",12}");
foreach (var item in controles)
{
    var mesure = TestAReference(item.seq);
    Console.WriteLine($"{item.label,-12} {mesure.chi2,9:F2} {mesure.p,12:G3}");
}

generateur        chi2   p (test A)


CA30  N=49      226.44        0.901


CA60  N=29     1739.81    4.89E-219


LFSR  n=17      234.81        0.813


LFSR  n=29      750.44     2.77E-50


LCG  2^31       609.38     4.88E-31


**Lecture du resultat** : le test A **separe** la regle 30 de trois des quatre temoins de
construction **lineaire**. Le CA60 est rejete avec une probabilite astronomiquement faible
(`4,9e-219`), le LCG aussi (`4,9e-31`), et le LFSR de longueur 29 aussi (`2,8e-50`), quand la regle
30 reste au centre (`0,901`). C'est la revendication centrale de Wolfram 1986 : les schemes lineaires
sont "provably random in some respects" et pourtant detectables, parce que leur structure algebrique
laisse une signature.

**Deux nuances que la mesure impose, et qu'il faut ecrire.**

1. Le **LFSR de longueur 17 passe** le test A (`0,813`, il n'est pas rejete). Wolfram publie pour lui
   `0,0390`, une valeur basse mais elle aussi non concluante a ce seul test : le rejet des LFSR vient
   de la **batterie**, pas d'un test isole. Un carnet qui conclurait "le test A rejette les LFSR" a
   partir d'une seule ligne serait en avance sur sa propre mesure.
2. Un `p` eleve n'est pas une preuve d'alea : il dit seulement que ce test-la ne voit rien. La
   section 6 ajoute un test qui, lui, est **decisif** sur la difference de nature entre une sequence
   lineaire et la regle 30.

## 5. Test C -- blocs distincts (et un resultat negatif assume)

Deuxieme procedure de l'Annexe A (p. 166) :

> C. Distinct Blocks Distribution. The frequencies with which p out of q successive m-blocks are
> distinct should follow a definite distribution. (m = 4, q = 4 are used.)

On groupe les bits en blocs de 4, on prend les groupes de 4 blocs successifs, on compte combien de
valeurs **distinctes** chacun contient, et on confronte la distribution observee a la distribution
exacte d'un tirage avec remise dans un alphabet de 16 symboles.

La distribution attendue se calcule exactement : la probabilite que `p` valeurs distinctes
apparaissent en `q` tirages vaut `(16)_p * S(q,p) / 16^q`, ou `(16)_p` est la factorielle
descendante et `S` le nombre de Stirling de seconde espece.

In [7]:
// Exercice 3 — test C : blocs distincts et loi theorique (stub compilable)
// TODO etudiant : renvoyer (chi2, p, obs).
//  - obs   : pour chaque groupe de 4 blocs successifs de 4 bits, compter le nombre de
//            valeurs distinctes ; les effectifs des groupes ayant 1, 2, 3 puis 4 blocs
//            distincts, dans cet ordre ;
//  - chi2  : ecart entre effectifs observes et effectifs attendus, la loi attendue etant
//            celle decrite dans la section 5 (tirage avec remise dans un alphabet de
//            16 symboles) ;
//  - p     : probabilite qu'une sequence vraiment aleatoire depasse ce chi-deux
//            (3 degres de liberte, comme dans le jumeau Python).
// L'oracle de reference du carnet (l'implementation de mesure) est fourni plus loin,
// dans la cellule de validation de la section 6 : comparez vos valeurs aux siennes,
// ligne a ligne -- l'ecart doit etre nul.
(double chi2, double p, int[] obs) TestC(int[] bits, int m = 4, int q = 4)
{
    // TODO etudiant : implementation attendue ici.
    return (0.0, 1.0, new int[] { 0, 0, 0, 0 });
}

Console.WriteLine("Exercice 3 a completer : test C (blocs distincts, chi-deux, p)");

Exercice 3 a completer : test C (blocs distincts, chi-deux, p)


**Lecture du resultat -- un test qui discrimine, mais pas les memes que le test A.** Le test C ne
rejette pas "les generateurs lineaires" en bloc : il rejette le **CA60** (`p = 0,0000`) et le
**LFSR de longueur 29** (`p = 0,0000`), et il laisse passer le **LFSR de longueur 17** (`p = 0,3712`)
et le **LCG** (`p = 0,2804`). La regle 30 passe (`p = 0,7708`).

Ce n'est pas une redite du test A : les deux tests ne voient pas les memes cas.

| Generateur | Test A | Test C | Rejete au moins une fois |
|---|---|---|---|
| CA30 (N=49) | passe (0,901) | passe (0,771) | **non** |
| CA60 (N=29) | rejete (4,9e-219) | rejete (0,0000) | oui |
| LFSR n=17 | passe (0,813) | passe (0,371) | **non** |
| LFSR n=29 | rejete (2,8e-50) | rejete (0,0000) | oui |
| LCG 2^31 | rejete (4,9e-31) | passe (0,280) | oui |

**Le cas qui compte est la ligne du LFSR de longueur 17** : il **echappe aux deux tests**. Un carnet
qui n'en aurait mesure qu'un, ou qui aurait conclu "les generateurs lineaires sont tous detectes",
serait passe a cote. Ce generateur est lineaire, sa sequence est par construction entierement
contenue dans 17 bits (section 6 le montrera), et pourtant ni la frequence des octets ni celle des
blocs distincts ne le voient sur 65 536 bits.

C'est exactement pourquoi l'Annexe A compte **sept** procedures et non deux. La these de Wolfram
(p. 159) dit "several of the tests used here", pas *all of them*, et sa propre Table 10.2 donne pour
ce generateur `0,0390` au test A -- une valeur basse, elle non plus non concluante. La mesure
reproduit donc aussi ce detail-la.

La mise en garde de l'Annexe A (p. 166) prend ici tout son sens :

> If the confidence interval is very close to zero or one, then the observed chi-deux is unlikely
> to be produced from a random sequence [...] Of course, if say a total of k tests are done, it is
> to be expected that the confidence interval for at least one of them will be less than 1/k.

Un resultat isole ne conclut pas, et **deux** non plus. Ce carnet en mesure deux sur sept, et le dit.

## 6. Complexite lineaire -- le test qui tranche sur la nature

Les tests A et C mesurent des **frequences**. Aucun des deux ne dit pourquoi les generateurs
lineaires doivent echouer quelque part : ils le constatent. La raison est structurelle, et elle se
mesure directement par la **complexite lineaire** : la longueur du plus court registre a decalage
theorique capable de reproduire la sequence (algorithme de Berlekamp-Massey).

C'est exactement l'argument que Wolfram oppose aux methodes lineaires (p. 162-163) :

> The primary feature of linear schemes is that they can be analysed by algebraic methods. As a
> consequence, certain randomness properties can be proved for the sequences they generate [...] But
> the simplicity in structure which underlies this analysis also limits the degree of randomness
> that such schemes can produce.

Une sequence de complexite lineaire `c` est **entierement determinee** par `2c` bits. Pour un LFSR de
longueur `n`, `c = n` : 17 ou 29 bits suffisent a reconstruire la sequence. Pour un generateur qui
n'est pas lineaire, `c` doit croitre avec la longueur echantillonnee, et se stabiliser autour de
`L/2` -- la valeur d'une sequence aleatoire.

In [8]:
// Validation — Test 9.9 de Wolfram (complexite lineaire) via Berlekamp-Massey sur GF(2) :
// longueur du plus court LFSR qui engendre la sequence. Lecture attendue --
// alea vrai : L croit comme n/2 (ici 2048 pour 4096 bits) ; LFSR n : L = n
// exactement (c'est sa definition) ; LCG : L proche de n/2, ce test ne le voit
// pas (c'est le test A qui l'a rejete ci-dessus -- chaque test voit une classe
// de defauts differente, d'ou la batterie).
// La meme cellule porte l'oracle de reference du test C de la section 5 : les valeurs
// lues dans la lecture de la section 5 sont produites ici, en execution reelle.

// --- Oracle de reference du test C (section 5) ---
// Nombres de Stirling de seconde espece S(n,p) pour p = 0..n (recurrence exacte).
long[] StirlingSecondeEspece(int n)
{
    var s = new long[n + 1];
    s[0] = 1;
    for (int k = 1; k <= n; k++)
    {
        var t = new long[n + 1];
        for (int j = 1; j <= k; j++)
            t[j] = j * s[j] + s[j - 1];
        s = t;
    }
    return s;
}

// Annexe A, test C : nombre de blocs distincts parmi q blocs m-bits successifs.
(double chi2, double p, int[] obs) TestCReference(int[] bits, int m = 4, int q = 4)
{
    int nm = bits.Length / m;
    var vals = new int[nm];
    for (int i = 0; i < nm; i++)
    {
        int v = 0;
        for (int j = 0; j < m; j++)
            v = (v << 1) | (bits[i * m + j] & 1);
        vals[i] = v;
    }
    int ng = vals.Length / q;
    var effectifs = new int[q + 1];
    for (int g = 0; g < ng; g++)
    {
        var distincts = new HashSet<int>();
        for (int k = 0; k < q; k++)
            distincts.Add(vals[g * q + k]);
        effectifs[distincts.Count]++;
    }
    // Loi attendue : P(p distincts sur q tirages) = (16)_p * S(q,p) / 16^q.
    int space = 1 << m;
    var stirling = StirlingSecondeEspece(q);
    var attendus = new double[q];
    for (int p = 1; p <= q; p++)
    {
        double descendante = 1.0;
        for (int k = 0; k < p; k++)
            descendante *= space - k;
        attendus[p - 1] = descendante * stirling[p] / Math.Pow(space, q);
    }
    double chi2 = 0.0;
    for (int p = 1; p <= q; p++)
    {
        double attendu = attendus[p - 1] * ng;
        double ecart = effectifs[p] - attendu;
        chi2 += ecart * ecart / attendu;
    }
    // 3 degres de liberte (q - 1), comme dans le jumeau Python.
    return (chi2, Chi2Survie(chi2, q - 1), effectifs.Skip(1).ToArray());
}

var sequencesControle = new (string label, int[] seq)[]
{
    ("CA30  N=49", TemporalBits(30, 49, L)),
    ("CA60  N=29", TemporalBits(60, 29, L)),
    ("LFSR  n=17", LfsrBits(17, new int[] { 14, 17 }, L)),
    ("LFSR  n=29", LfsrBits(29, new int[] { 27, 29 }, L)),
    ("LCG  2^31", LcgBits(L)),
};

Console.WriteLine("Test C (Annexe A) — oracle de reference :");
foreach (var item in sequencesControle)
{
    var r = TestCReference(item.seq);
    Console.WriteLine($"{item.label,-12} chi2={r.chi2,7:F2}  p={r.p:F4}  effectifs (1,2,3,4 blocs distincts)={string.Join(", ", r.obs)}");
}

Console.WriteLine();

// --- Berlekamp-Massey sur GF(2) ---
// Longueur du plus court LFSR engendrant la sequence.
int LineariteLineaire(int[] bits)
{
    var s = bits.Select(v => v & 1).ToArray();
    var c = new List<int> { 1 };
    var b = new List<int> { 1 };
    int longueur = 0;
    int m = 1;
    for (int i = 0; i < s.Length; i++)
    {
        int d = s[i];
        for (int j = 1; j <= longueur; j++)
        {
            if (j < c.Count)
                d ^= c[j] & s[i - j];
        }
        if (d == 1)
        {
            var precedent = new List<int>(c);
            // Le decalage de m zéros suivi de b (zero-padding de b a gauche).
            var decale = Enumerable.Repeat(0, m).ToList();
            decale.AddRange(b);
            while (c.Count < decale.Count)
                c.Add(0);
            for (int k = 0; k < decale.Count; k++)
                c[k] ^= decale[k];
            if (2 * longueur <= i)
            {
                longueur = i + 1 - longueur;
                b = precedent;
                m = 1;
            }
            else
            {
                m += 1;
            }
        }
        else
        {
            m += 1;
        }
    }
    return longueur;
}

int N_BITS = 4096;
Console.WriteLine($"{"generateur",-12} {"complexite lineaire",20}  (L/2 = {N_BITS / 2})");
foreach (var item in new (string label, int[] seq)[]
{
    ("CA30  N=49", TemporalBits(30, 49, N_BITS)),
    ("CA110 N=49", TemporalBits(110, 49, N_BITS)),
    ("LFSR  n=17", LfsrBits(17, new int[] { 14, 17 }, N_BITS)),
    ("LFSR  n=29", LfsrBits(29, new int[] { 27, 29 }, N_BITS)),
    ("LCG  2^31", LcgBits(N_BITS)),
})
{
    Console.WriteLine($"{item.label,-12} {LineariteLineaire(item.seq),20}");
}

Test C (Annexe A) — oracle de reference :


CA30  N=49   chi2=   1.13  p=0.7708  effectifs (1,2,3,4 blocs distincts)=2, 103, 1269, 2722


CA60  N=29   chi2= 303.58  p=0.0000  effectifs (1,2,3,4 blocs distincts)=6, 242, 1501, 2347


LFSR  n=17   chi2=   3.14  p=0.3712  effectifs (1,2,3,4 blocs distincts)=1, 88, 1249, 2758


LFSR  n=29   chi2= 145.37  p=0.0000  effectifs (1,2,3,4 blocs distincts)=11, 170, 1293, 2622


LCG  2^31    chi2=   3.83  p=0.2804  effectifs (1,2,3,4 blocs distincts)=0, 108, 1308, 2680


... (2 outputs left out)

CA110 N=49                   1674


LFSR  n=17                     17


LFSR  n=29                     29


LCG  2^31                    2051


**Lecture du resultat** : la separation est nette et elle est **de nature**, pas de degre. Les deux
LFSR ont une complexite lineaire egale a leur longueur (17 et 29) : toute leur sequence tient dans
une poignee de bits. La regle 30 mesuree sur 4 096 bits atteint `L/2`, la valeur que prend une
sequence aleatoire. Wolfram ne publie pas cette mesure dans l'article de 1986, mais il l'enonce en
prose : la simplicite algebrique des methodes lineaires **borne** ce qu'elles peuvent produire.

Une remarque d'honnetete, qui limite la portee de cette section : le LCG mesure ici une complexite
lineaire elevee (`L/2`), alors que le test A le rejetait. Les deux resultats ne se contredisent pas --
la complexite lineaire elevee d'un LCG est un fait connu, du a l'arithmetique modulaire -- mais ils
montrent que **chaque test voit une propriete differente**. C'est precisement pourquoi la batterie
de l'Annexe A en compte sept.

## 7. La reception -- ce que Gray oppose a la these

Wolfram ne se contente pas de proposer un generateur : il en fait un argument. Gray (2003, p. 203)
resume l'usage reel et pose la limite dans le meme paragraphe :

> the sequence of colors attained by the cell at the origin forms the basis for the random number
> generator used by Mathematica. This apparently perfectly random behavior was first noticed by
> Wolfram in 1984, and it seems to defy any sort of rigorous analysis. The best mathematical result
> that I know about Rule 30 is due to Erica Jen.

La these de Wolfram est **empirique** : la sequence passe les tests, donc elle est aleatoire "for
practical purposes" -- la formule exacte de la section 10 (p. 161) :

> The fact that deviations from randomness have not been found by any of the statistical procedures
> considered lends strong support to the belief that sequences produced by Eq. (3.1) with large N
> are indeed random for practical purposes.

Gray ne conteste pas les mesures ; il conteste le glissement de "passe les tests" a "est
aleatoire", et il le fait en deux endroits. Sur la regle 30 d'abord : il note que Wolfram croit
aussi la regle 30 **universelle**, et tranche -- "I find this to be sheer wishful thinking" (p. 203).
Sur la methode ensuite, en reprenant l'argument que Wolfram emploie contre les autres (p. 203) :
"unless a CA is obviously simple [...] it can simulate a universal Turing machine" est une these
generale que Gray juge non etayee.

Ce carnet mesure exactement ce que la these empirique permet de mesurer, et rien de plus :
**absence de deviation detectable par deux tests sur sept**, dans un regime ou la periode du
registre depasse l'echantillon. L'absence de detection n'est pas une preuve d'alea, et la section
10 de Wolfram le dit elle-meme (p. 158) : "one can never definitively conclude by studying a finite
sample that the complete sequence is not random".

## 8. La regle 110 -- la structure qui porte l'universalite

L'universalite de la regle 110 est un **theoreme**, pas une observation. Cook 2004 (p. 1) :

> The purpose of this paper is to prove a conjecture made by Stephen Wolfram in 1985, that an
> elementary one dimensional cellular automaton known as "Rule 110" is capable of universal
> computation.

Ce carnet ne re-prouve pas ce theoreme, et il ne peut pas le tester : l'universalite est une
propriete de la **famille** des configurations initiales, pas d'une trajectoire. Ce qu'il peut
mesurer, c'est la structure que la construction de Cook exploite -- et qui distingue la regle 110 de
la regle 30.

Cook (2004, section 2.3, p. 5) decrit les systemes de gliders comme le materiau de sa preuve :

> Each moving point is called a glider. [...] When two gliders meet, a lookup table specifies what
> the result of the collision should be.

La regle 110 possede un **fond periodique** (l'ether) sur lequel se deplacent des gliders. C'est ce
fond qui rend le codage possible : il fournit une memoire stable, ce qu'un regime chaotique ne
fournit pas. L'organe classe deja les deux regles ; mesurons la structure.

In [9]:
// Exercice 4 — diagramme espace-temps : une ligne par pas de temps (stub compilable)
// TODO etudiant : renvoyer le diagramme, une ligne par pas de temps, chaque ligne portant
// l'etat du registre (0/1) apres ce pas ; la graine est un site non nul unique, au centre du
// registre, comme dans le protocole de la section 1 ; l'automate est celui de l'organe
// (WolframStep), seule la mise en tableau est votre travail.
// L'ecriture attendue est rappelee dans la cellule de validation qui suit : un extrait de
// l'ether, une ligne par pas sur quatre, '#' pour un 1 et '.' pour un 0.
int[][] Spacetime(int regle, int nCells, int nSteps, int? seedSite = null)
{
    // TODO etudiant : implementation attendue ici.
    return Array.Empty<int[]>();
}

Console.WriteLine("Exercice 4 a completer : diagramme espace-temps (une ligne par pas de temps)");

Exercice 4 a completer : diagramme espace-temps (une ligne par pas de temps)


In [10]:
// Validation — periodes espace-temps de l'ether (regle 110) et temoin negatif (regle 30)
// Diagramme de reference : une ligne par pas de temps, construit avec l'automate de l'organe
// (WolframStep). C'est l'instrument de mesure du carnet ; l'exercice 4 en demande votre version.
int[][] SpacetimeReference(int regle, int nCells, int nSteps, int? seedSite = null)
{
    var state = SeedSingleNonZero(nCells, seedSite);
    var rows = new int[nSteps][];
    for (int t = 0; t < nSteps; t++)
    {
        rows[t] = state;
        state = WolframStep(state, regle);
    }
    return rows;
}

// Plus petites periodes (dt, dx) telles que pat[t+dt, x] == pat[t, x-dx] sur toute la fenetre.
List<(int dt, int dx)> SpacetimePeriod(int[][] pat, int a, int b, int t0, int t1, int maxDt = 14, int maxDx = 20)
{
    var found = new List<(int dt, int dx)>();
    for (int dt = 1; dt <= maxDt; dt++)
    {
        for (int dx = -maxDx; dx <= maxDx; dx++)
        {
            bool ok = true;
            for (int t = t0; t < t1 && ok; t++)
            {
                for (int x = a; x < b; x++)
                {
                    int xx = x - dx;
                    if (xx < 0 || xx >= pat[t].Length)
                    {
                        ok = false;
                        break;
                    }
                    if (pat[t + dt][x] != pat[t][xx])
                    {
                        ok = false;
                        break;
                    }
                }
            }
            if (ok)
                found.Add((dt, dx));
        }
    }
    return found;
}

// Plus petit decalage s tel que pat[t, x] == pat[t, x+s] sur toute la fenetre.
int? SpatialPeriod(int[][] pat, int a, int b, int t0, int t1, int maxShift = 40)
{
    for (int s = 1; s <= maxShift; s++)
    {
        bool ok = true;
        for (int t = t0; t < t1 && ok; t++)
        {
            for (int x = a; x < b - s; x++)
            {
                if (pat[t][x] != pat[t][x + s])
                {
                    ok = false;
                    break;
                }
            }
        }
        if (ok)
            return s;
    }
    return null;
}

// Classification d'une regle : densite et ecart-type sur les 100 derniers pas d'une
// trajectoire issue d'un etat initial aleatoire (graine 33), comme dans l'organe.
// Le generateur aleatoire du port n'est pas celui de NumPy : les indicateurs affiches sont
// la mesure propre de ce port ; l'etiquette de classe, elle, est indexee.
(string etiquette, double densite, double ecartType) ClassificationRegle(int regle, int nCells = 64, int nSteps = 500, ulong graine = 33)
{
    var rng = new SplitMix64(graine);
    var state = rng.Bits(nCells);
    var densites = new double[nSteps];
    for (int t = 0; t < nSteps; t++)
    {
        densites[t] = WolfrangDensite(state);
        state = WolframStep(state, regle);
    }
    var queue = densites.Skip(nSteps - 100).ToArray();
    double moyenne = queue.Average();
    double variance = queue.Select(v => (v - moyenne) * (v - moyenne)).Average();
    return (ClasseConnue(regle) ?? "non indexee", moyenne, Math.Sqrt(variance));
}

foreach (int r in new int[] { 30, 110 })
{
    var c = ClassificationRegle(r);
    Console.WriteLine($"regle {r,3} : {c.etiquette,-22} densite={c.densite:F4} ecart-type={c.ecartType:F4}");
}

var pat = SpacetimeReference(110, 600, 500);
Console.WriteLine();
Console.WriteLine("regle 110, extrait espace-temps (x de 250 a 400, un pas sur quatre) :");
for (int t = 440; t < 500; t += 4)
    Console.WriteLine(string.Concat(pat[t].Skip(250).Take(150).Select(v => v != 0 ? '#' : '.')));

var periodesEther = SpacetimePeriod(pat, 250, 380, 420, 480);
Console.WriteLine("periodes espace-temps de l ether (fenetre x=250..380, t=420..480) : [" + string.Join(", ", periodesEther.Select(p => $"({p.dt}, {p.dx})")) + "]");
Console.WriteLine("periode spatiale de l ether sur la meme fenetre : " + (SpatialPeriod(pat, 250, 380, 420, 480)?.ToString(CultureInfo.InvariantCulture) ?? "aucune (borne maxShift = 40 atteinte)"));

// Controle : la meme mesure sur la regle 30 ne doit rien trouver.
var pat30 = SpacetimeReference(30, 600, 500);
Console.WriteLine("meme mesure sur la regle 30 (temoin negatif) : [" + string.Join(", ", SpacetimePeriod(pat30, 250, 380, 420, 480).Select(p => $"({p.dt}, {p.dx})")) + "] / " + (SpatialPeriod(pat30, 250, 380, 420, 480)?.ToString(CultureInfo.InvariantCulture) ?? "aucune (borne maxShift = 40 atteinte)"));

// Contraste des regimes : persistance de la structure.
foreach (int r in new int[] { 30, 110 })
{
    var p = SpacetimeReference(r, 300, 400);
    var dens = p.Select(row => (double)row.Count(v => v != 0) / row.Length).ToArray();
    var queue = dens.Skip(dens.Length - 100).ToArray();
    double moyenne = queue.Average();
    double ecart = Math.Sqrt(queue.Select(v => (v - moyenne) * (v - moyenne)).Average());
    Console.WriteLine($"regle {r,3} : densite moyenne (100 derniers pas)={moyenne:F3} ecart-type={ecart:F4}");
}

regle  30 : III (chaotic)          densite=0.5008 ecart-type=0.0636


regle 110 : IV (complex)           densite=0.5742 ecart-type=0.0170


regle 110, extrait espace-temps (x de 250 a 400, un pas sur quatre) :


#...#..##.#####...#..##.#####...#..##.#####...#..##...................................................................................................


..#..##.#####...#..##.#####...#..##.#####...#..##.#...................................................................................................


#..##.#####...#..##.#####...#..##.#####...#..##.###...................................................................................................


... (13 outputs left out)

periode spatiale de l ether sur la meme fenetre : aucune (borne maxShift = 40 atteinte)


meme mesure sur la regle 30 (temoin negatif) : [] / aucune (borne maxShift = 40 atteinte)


regle  30 : densite moyenne (100 derniers pas)=0.501 ecart-type=0.0262


regle 110 : densite moyenne (100 derniers pas)=0.582 ecart-type=0.0112


**Lecture du resultat** : la fenetre choisie est invariante par les translations espace-temps
`(dt = 7, dx = 0)` et `(dt = 14, dx = 0)` -- le fond se repete **tous les 7 pas de temps**, a
position spatiale fixe. Le controle negatif est net : la meme mesure sur la regle 30 ne trouve
**aucune** translation, ni en temps ni en espace.

**Une note sur les etiquettes de classe.** Les etiquettes `III (chaotic)` et `IV (complex)` affichees
ici sont les libelles **indexes** de la table des classes de l'organe (le pendant .NET de
`WOLFRAM_CLASSES`) ; l'organe Python compose en plus indexation et mesure en
`III (known+measured)`. Les indicateurs affiches (densite, ecart-type) sont la **mesure propre de
ce port** : l'etat initial aleatoire vient du generateur SplitMix64(33) et non de NumPy, donc ils
different legerement de ceux du jumeau Python. Aucun des chiffres cites par la lecture ci-dessous
n'en depend : ils viennent tous de la partie deterministe (graine a site unique), qui est reproduite
a l'identique.

**Ce que la mesure ne dit pas, et qu'il faut ecrire.** La periode **spatiale** n'est pas etablie :
la fonction `SpatialPeriod` ne trouve aucun decalage invariant jusqu'a sa borne de recherche `max_shift = 40`, alors que la
litterature sur la regle 110 attribue au fond une periode spatiale de 14. Deux lectures sont
possibles et la mesure seule ne tranche pas : soit la fenetre `x = 250..380` contient autre chose
que du fond pur (un defaut, une frontiere de phase), soit l'invariance spatiale n'est pas
verifiable sur une bande de temps aussi courte dans un systeme ou des particules se deplacent.

Ce desaccord est instructif, et c'est pour cela qu'il est garde : une periode **espace-temps**
`(dt, dx)` est ce dont la construction de Cook a besoin -- c'est un repere qui se deplace, pas une
grille fixe. Chercher une periode purement spatiale revient a supposer que la structure est
immobile, ce qui est exactement l'hypothese que les gliders invalident.

C'est la difference que Cook exploite : un fond periodique est une **memoire**, une horloge. Un
glider qui s'y deplace peut coder un symbole, deux gliders qui se rencontrent peuvent implementer une
transition. Sur la regle 30, il n'y a rien a coder : le motif se defait, et la mesure le montre
aussi -- aucune translation, dans aucune direction.

La densite apporte un indice complementaire : la regle 110 garde une densite plus elevee et plus
stable que la regle 30 (`0,582` contre `0,501`, ecart-type `0,0112` contre `0,0262`). Mais il ne
faut pas surinterpreter ce nombre -- il mesure un remplissage, pas une structure. Le diagramme et
les periodes espace-temps sont les preuves ; la densite n'est qu'un indice.

## 9. Ce que le theoreme de Cook etablit -- et ce qu'il coute

Cook 2004 construit l'universalite en deux etages, et il les nomme dans son plan :

1. **section 2.2 (p. 4), les systemes de tags cycliques** : un systeme de reecriture sur bande finie,
   dont Cook montre qu'il est universel en simulant un systeme de tags ordinaire ;
2. **section 2.3 (p. 5), les systemes de gliders** : des particules mobiles dont les collisions
   obeissent a une table de correspondance -- et la regle 110 en fournit une implementation.

La conclusion (p. 1) est plus forte que "on peut calculer avec la regle 110" :

> implying that many questions concerning its behavior, such as whether a particular sequence of
> bits will occur, or whether the behavior will become periodic, are formally undecidable.

C'est le sens du mot *universel* ici : la regle 110 encode un modele de calcul, donc elle **herite
son indecidabilite**. Aucun test statistique ne peut etablir cela -- l'indecidabilite est un resultat
de theorie de la calculabilite, et le nier demande une preuve, pas une mesure.

Wolfram 1984 avait formule la conjecture au niveau des **classes**, avant la preuve (resume, p. 1) :

> The fourth class is probably capable of universal computation, so that properties of its infinite
> time evolution may be undecidable.

La difference de statut entre 1984 et 2004 est exactement celle que ce carnet met en scene : en
1984, la classe IV "est probablement" capable de calcul universel ; en 2004, une regle precise de
cette classe est **prouvee** universelle. Entre les deux, il n'y a pas plus de mesures -- il y a une
construction.

## 10. La reception de la regle 110 -- priorite, accord de non-divulgation, action juridique

Gray (2003, p. 203) raconte la chronologie, et son recit est le document de reception le plus precis
dont on dispose :

> The universal behavior of Rule 110 was first conjectured by Wolfram around 1985. The result was
> proved by Matthew Cook in the mid to late 1990s. However, Cook was prevented from publishing this
> result, because he had signed an agreement, as an employee of Wolfram Research, not to disclose
> his proof until ANKS was published. Because the appearance of ANKS was long delayed, Cook tested
> this agreement by presenting his result at a conference at the Santa Fe Institute, at which I was
> present. His talk was the highlight of the conference, but Wolfram threatened legal action,
> preventing it from appearing in the conference proceedings. Now that the nondisclosure agreement
> has expired, Cook's proof should become available.

Et son verdict sur la preuve elle-meme, qui n'est pas de complaisance :

> I have seen it and I highly recommend it, because it shows just how tenuous the original
> conjecture was -- the proof just barely works.

Deux lecons de reception, qui ne sont pas des anecdotes :

- **Un resultat peut etre vrai et retenu.** La priorite de la conjecture est a Wolfram (1985), la
  preuve a Cook (milieu-fin des annees 1990), la publication a 2004. Un lecteur qui citerait "Cook,
  universality de la regle 110" sans la date croirait a une decouverte publique de 1995.
- **La preuve est etroite.** "The proof just barely works" n'est pas une condamnation : c'est
  l'information la plus utile du paragraphe. Une universalite qui tiendrait avec une marge
  confortable serait une autre these ; celle qui tient de justesse dit quelque chose du regime de la
  classe IV.

## 10 bis. Portes de degenerescence -- le jouet peut-il porter le phenomene ?

Tout ce qui precede mesure. Reste une question que les sections 2 a 6 laissent ouverte, et qui vaut
pour n'importe quel jouet de la serie : **le modele est-il assez grand pour que le phenomene qu'on y
cherche puisse s'y produire ?** Un test qui ne rejette pas peut se tromper de deux facons opposees,
et les deux se sont deja presentees plus haut :

- un `p` central sur un registre dont la periode tient dans l'echantillon ne dit pas "aleatoire", il
  dit "la meme boucle repassee plusieurs fois" -- c'est le cas `N=23` de la section 3 ;
- un `p` central sur un automate trivial ne dit rien non plus : il n'y a simplement rien a detecter.

Une porte se pose donc en trois temps, dans le vocabulaire du protocole de degenerescence de la
serie (issue #20300) :

**G1 -- temoin positif : le phenomene peut apparaitre.** Sur le jouet retenu, la mesure doit tomber
dans la zone ou l'instrument est calibre. Si elle n'y tombe pas, le jouet est mal pose et aucune
conclusion negative n'est disponible.

**G2 -- temoin negatif : le phenomene n'est pas produit par construction.** Un jouet construit pour
etre trivial doit etre **rejete** par la meme mesure. Sans ce temoin, un test qui n'a jamais rien
rejete serait indiscernable d'un test casse.

**G3 -- comparaison d'echelle : la lecture ne depend pas de la taille par accident.** Le meme jouet,
mesure a plusieurs tailles, doit rendre un verdict stable -- ou bien l'echelle fait partie de la
conclusion.

Le critere de degenerescence retenu ici n'est **pas** la valeur de `p` : c'est la **periode du
registre comparee a la longueur de l'echantillon**. C'est le seul des deux qui parle de la
dynamique ; `p` ne parle que de l'instrument, et la section 3 a deja montre qu'il peut se taire alors
que la degenerescence est totale.

Les trois temoins negatifs sont des regles qui ne peuvent rien engendrer : `0` (constante nulle),
`255` (constante un) et `170` (decalage pur). Aucune n'a de dynamique propre -- si la porte ne les
rejetait pas, elle ne vaudrait rien.

In [11]:
// Exercice 5 — portes de degenerescence : un verdict par jouet, derive de la PERIODE, jamais de p.
// G1 temoin positif | G2 temoins negatifs | G3 comparaison d'echelle.
// TODO etudiant : Porte doit rendre un verdict par jouet, derive de la PERIODE comparee a
// L_PORTE -- et jamais de la p. La regle de verdict est celle enoncee en section 10 bis :
// une periode superieure a L_PORTE (ou non atteinte dans la fenetre de recherche) laisse le
// jouet SAIN ; une periode inferieure ou egale a L_PORTE le rend DEGENERE.
// TODO etudiant : la periode se lit dans l'oracle PeriodeReference (section 3) -- votre
// version `Periode` (exercice 1) doit rendre la meme valeur ; la p se lit dans l'oracle
// TestAReference (section 3) -- elle est affichee pour information, elle n'entre pas
// dans le verdict.
int L_PORTE = 64 * 1024;
const int MAX_PAS_PORTE = 300000;

string Porte(int regle, int nCells, int maxSteps = MAX_PAS_PORTE)
{
    // TODO etudiant : implementation attendue ici.
    return "a completer";
}

var jouets = new (string nom, int regle, int nCells)[]
{
    ("G1  regle  30  N=49", 30, 49),
    ("G2  regle   0  N=49", 0, 49),
    ("G2  regle 170  N=49", 170, 49),
    ("G2  regle 255  N=49", 255, 49),
    ("G3  regle  30  N=17", 30, 17),
    ("G3  regle  30  N=23", 30, 23),
    ("G3  regle  30  N=29", 30, 29),
    ("G3  regle  30  N=37", 30, 37),
};

Console.WriteLine($"{"jouet",-22} {"p (test A)",11} {"periode",9}  verdict");
Console.WriteLine(new string('-', 74));
var resultats = new List<(string nom, string verdict)>();
foreach (var jouet in jouets)
{
    var mesure = TestAReference(TemporalBits(jouet.regle, jouet.nCells, L_PORTE));
    var (periode, _) = PeriodeReference(jouet.regle, jouet.nCells);
    string verdict = Porte(jouet.regle, jouet.nCells);
    resultats.Add((jouet.nom, verdict));
    Console.WriteLine($"{jouet.nom,-22} {mesure.p,11:E4} {(periode.HasValue ? periode.Value.ToString(CultureInfo.InvariantCulture) : "non mes."),9}  {verdict}");
}

Console.WriteLine();
Console.WriteLine("note : les periodes viennent de l'oracle PeriodeReference (section 3) ; les verdicts viennent du stub Porte (exercice 5), a completer.");
int nDeg = resultats.Count(r => r.verdict.StartsWith("DEGENERE", StringComparison.Ordinal));
Console.WriteLine();
Console.WriteLine($"jouets degeneres : {nDeg} sur {resultats.Count}");
Console.WriteLine($"verdict du jouet retenu (regle 30, N=49) : {resultats[0].verdict}");


jouet                   p (test A)   periode  verdict


--------------------------------------------------------------------------


G1  regle  30  N=49    9.0070E-001  non mes.  a completer


G2  regle   0  N=49    0.0000E+000         1  a completer


G2  regle 170  N=49    0.0000E+000        49  a completer


G2  regle 255  N=49    0.0000E+000         1  a completer


... (3 outputs left out)

G3  regle  30  N=37    5.3248E-001  non mes.  a completer


note : les periodes viennent de l'oracle PeriodeReference (section 3) ; les verdicts viennent du stub Porte (exercice 5), a completer.


jouets degeneres : 0 sur 8


verdict du jouet retenu (regle 30, N=49) : a completer


**Lecture du resultat -- les trois portes, et celle qui tranche.**

Les periodes de la colonne `Periode` sont produites par l'oracle `PeriodeReference` (section 3),
qui sert de reference a l'exercice 1 ; les verdicts de la colonne `Verdict` sont ceux que rend le
stub `Porte` (exercice 5), que l'apprenant doit completer -- la lecture qui suit en donne les
valeurs attendues.

| Porte | Jouet | `p` (test A) | Periode | Verdict |
|---|---|---|---|---|
| G1 | regle 30, `N=49` | 0,9007 | non mesuree (aucune repetition sur 300k pas) | **SAIN** |
| G2 | regle 0 (constante nulle) | 0,0000 | 1 | DEGENERE |
| G2 | regle 170 (decalage pur) | 0,0000 | 49 | DEGENERE |
| G2 | regle 255 (constante un) | 0,0000 | 1 | DEGENERE |
| G3 | regle 30, `N=17` | 2,0e-10 | 10 846 | DEGENERE |
| G3 | regle 30, `N=23` | **0,8253** | 38 249 | **DEGENERE** |
| G3 | regle 30, `N=29` | 0,2556 | non mesuree (aucune repetition sur 300k pas) | SAIN |
| G3 | regle 30, `N=37` | 0,5325 | non mesuree (aucune repetition sur 300k pas) | SAIN |

La colonne `Periode` distingue deux etats et pas trois : un nombre, ou `non mesuree (aucune repetition sur 300k pas)` --
la sortie de la cellule precedente dit `non mes.` pour ces trois lignes. Une periode plus longue que
la fenetre d'observation n'est pas mesuree ; elle est seulement non atteinte, et l'ecrire comme un
nombre la ferait passer pour un resultat.

**G1 -- le temoin positif tient.** La regle 30 a `N=49` tombe a `p = 0,9007`, au centre de la zone
calibree : le jouet retenu peut porter le phenomene, et la question "y a-t-il quelque chose a
mesurer ?" est fermee. C'est la seule ligne dont on ait besoin pour rendre les autres lisibles.

**G2 -- les trois temoins negatifs sont rejetes, et c'est tout ce qu'on leur demande.** La constante
nulle (periode 1), la constante un (periode 1) et le decalage pur (periode 49) rendent tous
`p = 0,0000` : la porte ne laisse passer aucun des jouets construits pour ne rien produire. Sans ce
resultat, un `p` central sur la regle 30 serait indistinguable d'un instrument qui n'a jamais rien
rejete -- c'est-a-dire d'un instrument mort.

**G3 -- sur un cas sur quatre, la porte attrape ce que `p` laisse passer.** Les quatre tailles se
separent en deux groupes :

- `N=17` (6,04 tours de cycle dans l'echantillon) et `N=23` (1,71 tour) sont **degeneres** : la
  sequence mesuree n'est pas 65 536 pas de dynamique, c'est la meme boucle repassee ;
- `N=29` et `N=37` ne montrent **aucune repetition d'etat dans la fenetre de 300 000 pas**.
  Ce n'est pas la meme chose qu'une periode plus longue que la fenetre : `Periode` s'arrete
  a `maxSteps` et rend `(null, null)`, ce qui ne distingue pas un cycle long d'une **longue
  preperiode**. Un contre-exemple suffit : sur les entiers, `f(x) = min(x + 1, 300001)` part
  de 0, ne repasse par aucun etat en 300 000 pas, et sa periode vaut pourtant 1. Le verdict de
  ces deux lignes est donc **SAIN au sens d'un controle borne de non-repetition observee** --
  pas une borne prouvee sur la periode.

Un seul cas sur quatre justifie la porte : `N=17` est rejete par `p` tout seul (`2,0e-10`), et `N=29`
comme `N=37` sont sains par les deux instruments -- la porte n'y ajoute rien. C'est `N=23` qui la
justifie. Son `p` vaut **0,8253** : le test A ne rejette pas, et une lecture qui s'arreterait la
conclurait "rien a signaler". La periode dit l'inverse -- 38 249 pas pour un echantillon de 65 536,
soit 1,71 tour -- et la frequence des octets y est donc calculee sur une sequence ou chaque motif
revient presque deux fois. **Un `p` central ne certifie pas que le jouet est sain ; il certifie
seulement que ce test-la ne voit rien.** C'est la lecon de la section 3 transformee en instrument :
elle y etait enonciation, elle est ici un verdict calcule.

**Ce que la porte etablit, et ce qu'elle n'etablit pas.** Elle etablit, **sur les cinq tailles
testees** (`N` = 17, 23, 29, 37, 49), que le jouet
automate cellulaire **porte** le phenomene de `N=29` a `N=49`, et qu'il ne le porte ni a
`N=23` ni a `N=17` -- ce qui donne un contenu mesurable a la regle pratique de Wolfram
(`N=49` pour depasser `2^32`, section 11 p. 162). Elle ne prouve pas que 29 soit un seuil :
les tailles intermediaires n'ont pas ete mesurees. Elle n'etablit pas que la regle 30 est
aleatoire : une periode non mesuree n'est pas une
periode infinie, et deux tests sur sept ne totalisent pas une batterie. Elle ne dit rien non plus de
la regle 110, dont le regime est structure et non chaotique -- le jouet "universalite" demanderait sa
propre porte, et Cook 2004 montre qu'elle ne serait pas statistique.

Une derniere limite, qui vaut pour toute la serie : la porte mesure la **taille** du modele, pas la
**fidelite** du protocole. Le desaccord chiffre de la section 3 sur la Table 10.1 reste entier -- les
trois degres de liberte du protocole publie (site lu, position de la graine, ordre des bits dans
l'octet) ne sont toujours pas fixes.

## 11. Synthese -- quatre statuts, et ce que ce carnet a reellement etabli

| Affirmation | Source | Statut | Mesure ici |
|---|---|---|---|
| La regle 30 produit des sequences sans deviation detectable en regime non periodique | Wolfram 1986, p. 159-161 | these empirique | **reproduite** (test A et test C, `N >= 29`) |
| Les deviations apparaissent quand la periode du registre approche la longueur echantillonnee | Wolfram 1986, p. 159 | enonce conditionnel | **reproduite**, et **durcie** par la porte de la section 10 bis : a 1,7 tour le test ne detecte rien alors que l'instance est deja degeneree -- l'absence de detection n'est pas un acquittement |
| Les LFSR et les LCG sont detectes par la batterie | Wolfram 1986, p. 159-160 | these empirique | **reproduite en partie** (A et C rejettent CA60, LFSR n=29 et LCG ; le LFSR n=17 echappe aux deux) |
| Les generateurs lineaires ont une structure identifiable | Wolfram 1986, p. 162-163 | argument structurel | **reproduite** (complexite lineaire : 17 et 29 pour les LFSR, `L/2` pour la regle 30) |
| Les valeurs exactes de la Table 10.1 sont reproductibles | Wolfram 1986, p. 159 | -- | **non reproduites** (protocole sous-specifie : site, position de graine, ordre des bits) |
| La regle 110 est universelle | Cook 2004, p. 1 | **theoreme** | non mesurable -- cite, non teste |
| La classe IV est "probablement" universelle | Wolfram 1984, resume | conjecture | encadree par le theoreme de 2004 |
| La regle 30 elle-meme serait universelle | Gray 2003, p. 203 | opinion de Wolfram, contestee | non mesuree -- "sheer wishful thinking" selon Gray |

La colonne qui compte est la troisieme. Un carnet qui traiterait les huit lignes de la meme facon
serait faux sur au moins cinq.

La porte de la section 10 bis ne reproduit aucune affirmation de plus : elle produit la seule de ce
carnet qui ne vienne d'aucune source. Sa conclusion -- sur les cinq tailles testees, le jouet
porte le phenomene de `N=29` a
`N=49` et ne le porte pas aux deux tailles inferieures -- est un resultat **de ce carnet**,
obtenu en croisant deux mesures que les sections 2 et 3 presentaient separement. Elle est
bornee a ces instances : rien n'y etablit que 29 soit un seuil, et pour les grandes tailles
elle n'etablit pas la periode mais seulement l'**absence de repetition dans la fenetre
observee**. C'est aussi la seule ligne du carnet qui pourrait etre
fausse sans que personne ne s'en apercoive : les autres ont un texte en regard.

## Vérifiez votre compréhension

1. Le test A et le test C reposent sur `N >= 29` — pourquoi ce seuil borne-t-il leur portée ?
2. La complexité linéaire distingue la règle 30 des LFSR (17 et 29 contre `L/2`) — que mesure exactement cet écart ?
3. La porte de la section 10 bis étend le phénomène de `N=29` à `N=49` — qu'établit-elle, et qu'est-ce qu'elle n'établit pas ?

## Exercice A -- Sensibilite du protocole : le site echantillonne

Le carnet a fixe `site=0` et une graine au centre du registre, parce que le texte de 1986 ne les
precise pas. Cet exercice mesure l'effet de ce choix.

**Objectif** : pour `N = 17` et `L = 64k`, calculer le `p` du test A pour trois sites (0, `N//4`,
`N//2`) et trois positions de graine (0, `N//2`, `N - 1`). Remplir un tableau 3x3 et dire si le
verdict (rejet ou non) change.

**Indice** : `TemporalBits(30, N, L, site: s, seedSite: g)` accepte les deux parametres.

**Question** : si le verdict change selon le site, que peut-on encore affirmer sur "la regle 30 a
N=17" ?

In [12]:
// Exercice A — grille de sensibilite du protocole (stub compilable)
// TODO etudiant : remplir la grille des p du test A.
// resultat[i, j] : le p du test A pour le site sites[i] et la position de graine graines[j] ;
// une case reste a null tant qu'elle n'est pas calculee. Le verdict associe est
// « rejet » si p < 0,05 ou p > 0,95, « conforme » sinon.
int N = 17;
int Lx = 64 * 1024;
var sites = new int[] { 0, N / 4, N / 2 };
var graines = new int[] { 0, N / 2, N - 1 };

// TODO etudiant : implementation attendue ici.
double?[,] resultat = new double?[3, 3];

// Etape 1 : boucler sur les sites et les graines
// Etape 2 : appeler TemporalBits(30, N, Lx, site: s, seedSite: g) puis TestAReference
// Etape 3 : afficher le verdict (rejet si p < 0.05 ou p > 0.95) pour chaque case
Console.WriteLine("Exercice A a completer : grille de sensibilite du protocole");

Exercice A a completer : grille de sensibilite du protocole


## Exercice B -- La regle 90 est-elle lineaire, et le voit-on ?

La regle 90 (XOR du voisin gauche et du voisin droit) engendre le triangle de Sierpinski : un motif
visuellement spectaculaire, mais produit par une operation **lineaire**.

**Objectif** : mesurer la complexite lineaire de la regle 90 sur 4 096 bits, et la comparer a celle
de la regle 30 et a celle d'un LFSR.

**Indice** : `LineariteLineaire(TemporalBits(90, 49, 4096))`.

**Question** : si la regle 90 a une complexite lineaire basse, que cela implique-t-il pour un
carnet qui presenterait son triangle comme une preuve de "complexite" ?

In [13]:
// Exercice B — complexite lineaire de la regle 90 (stub compilable)
// TODO etudiant : mesurer la complexite lineaire de la regle 90 sur 4 096 bits, puis celle de
// la regle 30 comme reference. L'instrument (Berlekamp-Massey sur GF(2)) est deja fourni par
// la cellule de validation de la section 6 : LineariteLineaire(sequence).
// Les valeurs ne sont pas donnees ici : comparez-les a celles de la section 6.
int? lc90 = null; // TODO : complexite lineaire de la regle 90
int? lc30 = null; // TODO : complexite lineaire de la regle 30 (reference)

// Etape 1 : extraire 4096 bits de la regle 90 avec TemporalBits
// Etape 2 : passer la sequence a LineariteLineaire
// Etape 3 : comparer aux valeurs de reference de la section 6
if (lc90 is null && lc30 is null)
    Console.WriteLine("Exercice B a completer : complexite lineaire de la regle 90");
else
    Console.WriteLine($"regle 90 : {lc90} ; regle 30 : {lc30}");

Exercice B a completer : complexite lineaire de la regle 90


## Exercice C -- Mesurer la periode de l'ether de la regle 110 sur une autre fenetre

La section 8 a mesure les translations espace-temps `(dt = 7, dx = 0)` et `(dt = 14, dx = 0)` sur la
fenetre `x = 250..380`, et n'a trouve **aucune** invariance purement spatiale. Cet exercice verifie
que ces deux resultats ne sont pas des artefacts de la fenetre choisie.

**Objectif** : refaire la mesure sur deux autres fenetres (par exemple `x = 200..320` et
`x = 300..420`), puis sur un diagramme plus large (`n_cells = 800`, en gardant `n_steps = 500`).

**Indice** : `SpacetimeReference(110, 800, 500)` puis `SpacetimePeriod(...)` et `SpatialPeriod(...)`.

**Question** : si la periode temporelle reste 7 sur toutes les fenetres alors que l'invariance
spatiale ne se trouve nulle part, que mesure-t-on reellement -- le fond de l'automate, ou le fait que
les particules qui s'y deplacent cassent toute periodicite spatiale ?

In [14]:
// Exercice C — robustesse de la periode de l'ether (stub compilable)
// TODO etudiant : liste des periodes trouvees par fenetre ; une entree par fenetre mesuree.
// Fenetres a tester : par exemple x = 200..320 et x = 300..420, puis un diagramme plus large
// avec nCells = 800 (nSteps = 500 reste inchange). A chaque fenetre, relever les translations
// espace-temps invariantes et l'existence d'une invariance purement spatiale ; comparer
// ensuite avec le couple (7, 0) trouve sur la fenetre x = 250..380.
// L'instrument de mesure (construction du diagramme et recherche des translations) est
// fourni par la cellule de validation de la section 8.
var results = new List<string>(); // TODO : liste des periodes trouvees par fenetre

// Etape 1 : construire un diagramme plus large avec SpacetimeReference(110, 800, 500)
// Etape 2 : appliquer SpacetimePeriod sur deux fenetres disjointes
// Etape 3 : comparer avec (7, 0) et commenter
Console.WriteLine("Exercice C a completer : robustesse de la periode de l ether");

Exercice C a completer : robustesse de la periode de l ether


## À retenir

*Carnet d'origine ICT (pli 2 de l'Origami Wolfram, anciennement ICT-50), rattaché à la série
Complexity — voir le [registre des renommages](../../../docs/reference/rename-ledger.tsv).*

*Ce carnet ferme les deux points de la definition du pli 2 que la serie ne couvrait pas, et il les
ferme avec des statuts differents, ce qui est le fond du sujet :*

1. **La regle 30 comme generateur** est une these **empirique**. Elle se mesure, et la mesure
   reproduit ce qui est reproductible : l'absence de deviation detectable quand la periode du
   registre depasse l'echantillon, et le rejet des generateurs lineaires par une batterie de tests
   -- avec la complexite lineaire comme discriminant de nature. Elle ne reproduit pas les valeurs
   exactes de la Table 10.1, et le carnet dit pourquoi plutot que de choisir un jeu de parametres
   qui tombe juste.
2. **L'universalite de la regle 110** est un **theoreme**. Il ne se mesure pas. Ce qui se mesure,
   c'est la structure qui le rend possible -- le fond periodique de periode 7 en temps, 14 en
   espace -- et cette mesure explique pourquoi la classe IV peut coder la ou la classe III ne le
   peut pas.

**Limites assumees** : deux tests sur les sept de l'Annexe A ; un seul site echantillonne ; pas de
reproduction des periodes de la Table 9.2 au-dela de `N = 23` (fenetre de recherche bornee a
300 000 pas) ; la preuve de Cook n'est ni lue integralement ni verifiee, seulement citee.

**References** (copies archivees, `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\`) :

- Wolfram, S. (1986). *Random Sequence Generation by Cellular Automata*. Advances in Applied
  Mathematics 7, 123-169. Section 10 (p. 158-161), section 11 (p. 161-163), Annexe A (p. 166-167).
- Wolfram, S. (1984). *Universality and Complexity in Cellular Automata*. Physica D 10, 35-57.
  Resume (p. 1), definition des quatre classes (p. 5).
- Cook, M. (2004). *Universality in Elementary Cellular Automata*. Complex Systems 15(1), 1-40.
  Section 1 (p. 1-2), sections 2.2-2.3 (p. 4-5).
- Gray, J. (2003). *A Mathematician Looks at Wolfram's New Kind of Science*. Notices of the AMS
  50(2), 200-211. Regle 30 et generateur de Mathematica (p. 203), priorite de la regle 110 et
  accord de non-divulgation (p. 203).

## Pour aller plus loin

- **[Complexity-08b — Combinateurs SKI et tag de Post](Complexity-08b-CombinateursSKI-TagDePost-Python.ipynb)** : le carnet suivant de l'accrétion, où la réception critique porte sur les combinateurs et le problème ouvert du tag de Post.
- **[ICT-0 — Cadrage de la série](../../IIT/ICT-Series/ICT-0-Framing.md)** : le principe directeur de la série ICT d'origine.

> Carnet issu de l'EPIC #26534.